# Explainability

Explainability review across trained models.

Steps:
- Locate explainability artifacts.
- Review feature importance and saliency outputs.
- Run the claim evidence report.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path

summary = {
    'plots': [],
    'saliency': {},
}


def run_optional(cmd: list[str]) -> int:
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env)
    print('Return code:', result.returncode)
    return result.returncode


In [ ]:
# Locate explainability artifacts.
plots_root = REPO_ROOT / 'experiments'
plot_files = []
if plots_root.exists():
    for path in plots_root.rglob('*'):
        if path.is_file() and path.parent.name == 'plots':
            plot_files.append(path)

summary['plots'] = [str(p.relative_to(REPO_ROOT)) for p in plot_files[:20]]
print('Found plot files:', len(plot_files))
for item in summary['plots']:
    print(' -', item)


In [ ]:
# Review saliency outputs if available.
saliency_path = REPO_ROOT / 'experiments' / 'behavior' / 'plots' / 'saliency.csv'
if saliency_path.exists():
    df = pd.read_csv(saliency_path)
    print('Saliency shape:', df.shape)
    summary['saliency']['columns'] = list(df.columns)
    print(df.head(10))
else:
    print('Missing saliency.csv')


In [ ]:
# Run the claim evidence report.
script_path = REPO_ROOT / 'scripts' / 'claim_evidence.py'
if script_path.exists():
    run_optional([PY, 'scripts/claim_evidence.py'])
else:
    print('Missing:', script_path)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_explainability_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
